## 2026 EY AI & Data Challenge - TerraClimate Data Extraction Notebook

This notebooks access and extracts data from the TerraClimate dataset. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format. 

For more information, visit: [terraclimate- overview](https://planetarycomputer.microsoft.com/dataset/terraclimate#overview) 

In [ ]:
!pip install uv
!uv pip install  -r ../requirements.txt 

In [1]:
import warnings
warnings.filterwarnings("ignore")

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

from scipy.spatial import cKDTree

from datetime import date
from tqdm import tqdm

# Importing functions
import sys
import os
sys.path.append(os.path.abspath('..'))
from utils import save_df, get_catalog

In [ ]:
# Selected features
selected_features = [
    'pet', 'ppt', 'q', 'tmax', 'tmin', 'srad', 'soil', 'ws'
]

# Save path
save_path = 'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/data/intermediate/'

# MS Planetary Catalog Collection
catalog = get_catalog(
    'https://planetarycomputer.microsoft.com/api/stac/v1'
)

# Column mapping
col_mapping = {
    'lat': 'Latitude',
    'lon': 'Longitude',
    'time': 'Sample Date'
}

In [5]:
# Dataframe for training
water_quality_df = (
    pd.read_csv('../data/raw/water_quality_training_dataset.csv')
)

# Dataframe for testing
validation_df = (
    pd.read_csv('../data/raw/submission_template.csv')
)

### Functions

In [2]:
def load_terraclimate_dataset():
    '''
    Opens the TerraClimate Zarr/NetCDF dataset from the
    Microsoft Planetary Computer, handling storage options
    automatically.
    '''
    collection = catalog.get_collection('terraclimate')
    asset = collection.assets['zarr-abfs']

    # Takes the storage options if it exists
    if 'xarray:storage_options' in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields['xarray:storage_options'],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(
            asset.href,
            **asset.extra_fields['xarray:open_kwargs'],
        )

    return ds

In [3]:
def filterg():
    '''
    Filters the dataset for the desired time range (2011–2015) and the
    spatial extent corresponding to the study region. The resulting data
    is converted into a pandas DataFrame with standardized column names.
    '''
    # Opens connection
    ds = load_terraclimate_dataset()
    
    # Filtered by date dataframe
    ds_subset = (
        ds[selected_features]
        .sel(
            time=slice('2011-01-01', '2015-12-31')
        )
    )

    # Lat + Lon limits (I adjusted a little to include borders)
    # My slice is infinitely more efficient, but exclude borders
    lat_min, lat_max = -35.22, -21.68
    lon_min, lon_max = 14.93, 32.83

    # Slices by lat + lon
    # Better than downloading every pixel and filtering (xarray's lazy loading)
    ds_geo = (
        ds_subset
        .sel(
            lat=slice(lat_max, lat_min),
            lon=slice(lon_min, lon_max)
        )
    )

    # To memory
    df_all = (
        ds_geo
        .to_dataframe()
        .reset_index()
        .dropna(subset=selected_features)
    )

    # Final df
    df_var_final = df_all.rename(columns=col_mapping)

    return df_var_final

In [4]:
def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Map nearest climate variable values to a new DataFrame 
    containing only the specified variable column.
    """
    # Copying dataset and adding index
    sa_temp = (
        sa_df[['Latitude', 'Longitude', 'Sample Date']]
        .copy()
    )
    sa_temp['_orig_idx'] = sa_temp.index

    # Removing duplicates from climate_df
    climate_locs = (
        climate_df[['Latitude', 'Longitude']]
        .drop_duplicates()
    )

    # Coords
    sa_coords = np.radians(sa_temp[['Latitude', 'Longitude']])
    climate_coords = np.radians(climate_locs[['Latitude', 'Longitude']])

    # Searching for closest spatial neibourhood
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)

    # Climatic grid for grouping
    sa_temp['grid_lat'] = climate_locs.iloc[idx]['Latitude'].values
    sa_temp['grid_lon'] = climate_locs.iloc[idx]['Longitude'].values

    # Guarantees datetime and order by it - Merge AsOf
    sa_temp['Sample Date'] = pd.to_datetime(sa_temp['Sample Date'], dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')
    sa_temp = sa_temp.sort_values('Sample Date')
    climate_sorted = climate_df.sort_values('Sample Date')

    # Join the dfs by nearest date, on equal lat + lon
    merged = pd.merge_asof(
        sa_temp, 
        climate_sorted[['Latitude', 'Longitude', 'Sample Date', var_name]], 
        on='Sample Date',
        left_by=['grid_lat', 'grid_lon'],
        right_by=['Latitude', 'Longitude'],
        direction='nearest'
    )

    # Original sort
    merged = merged.sort_values('_orig_idx')

    return pd.DataFrame({
            var_name: merged[var_name].values
        },
        index=merged['_orig_idx']
    )
    

In [ ]:
def process_df(destination_df, original_df):
    '''
    Just 'unions' the databases
    '''
    # Finalizing df
    destination_df['Latitude'] = original_df['Latitude']
    destination_df['Longitude'] = original_df['Longitude']
    destination_df['Sample Date'] = original_df['Sample Date']
    cols_to_keep = [
        'Latitude', 'Longitude', 'Sample Date'
    ] + selected_features
    
    return destination_df[cols_to_keep]

## Data transformation

In [7]:
# Load TerraClimate dataset, filter (time,region,parameter),
# filter for nearest parameter values
climate_master_df = filterg()

In [ ]:
terraclimate_training_df_final = pd.DataFrame()
terraclimate_validation_df_final = pd.DataFrame()

for var_name in selected_features:
    print(f"Mapping for {var_name}")
    
    terraclimate_training_df = assign_nearest_climate(water_quality_df, climate_master_df, var_name)
    terraclimate_training_df_final[var_name] = terraclimate_training_df[var_name]
    
    terraclimate_validation_df = assign_nearest_climate(validation_df, climate_master_df, var_name)
    terraclimate_validation_df_final[var_name] = terraclimate_validation_df[var_name]

In [8]:
terraclimate_training_df_processed = process_df(terraclimate_training_df_final, water_quality_df)
terraclimate_validation_df_processed = process_df(terraclimate_validation_df_final, validation_df)

## Data saving

In [ ]:
save_df(terraclimate_training_df_processed, 'terraclimate_train_features_base', save_path)
save_df(terraclimate_validation_df_processed, 'terraclimate_val_features_base', save_path)